In [8]:
from rich import print as rprint

import torch as t
from transformers import GPT2LMHeadModel, GPT2Tokenizer
# from nnsight import NNsight
from nnterp import StandardizedTransformer

import circuitsvis as cv

In [10]:
device = "cuda" if t.cuda.is_available() else "cpu"

MODEL_NAME = "gpt2"

In [3]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [4]:
rprint(model.config)

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.6",
  "use_cache": true,
  "vocab_size": 50257
}

In [5]:
tokens = tokenizer("A cat was", return_tensors="pt")
tokens = {k: v.to(device) for k, v in tokens.items()}

rprint(tokens)

{'input_ids': tensor([[  32, 3797,  373]]), 'attention_mask': tensor([[1, 1, 1]])}

In [12]:
# tokenizer.pad_token = tokenizer.eos_token

# sentences = [
#     "Hello world",           # 2 tokens
#     "Tokenization is cool"   # 4 tokens
# ]

# # padding=True forces the shorter sentence to match the length of the longest one
# tokens = tokenizer(sentences, padding=True, return_tensors="pt")

# rprint("Input IDs:\n", tokens["input_ids"])
# rprint("\nAttention Mask:\n", tokens["attention_mask"])

In [11]:
output = model(**tokens, labels=tokens['input_ids'])

rprint('loss =', output.loss)

probabilities = output.logits.softmax(dim=-1)
rprint(tokenizer.decode(probabilities.argmax(dim=-1).squeeze()))

loss = tensor(6.3620, grad_fn=<NllLossBackward0>)

. was found

##### how many tokens does your model guess correctly
https://arena-chapter1-transformer-interp.streamlit.app/~/+/[1.2]_Intro_to_Mech_Interp#exercise-how-many-tokens-does-your-model-guess-correctly

In [12]:
test_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

test_tokens = tokenizer(test_text, return_tensors='pt')
out = model(**test_tokens, labels=test_tokens['input_ids'])
rprint(f'loss = {out.loss}')

probabilities = out.logits.softmax(dim=-1)
predictions = probabilities.argmax(dim=-1)
predictions = t.cat([t.tensor([[0]], device=device), predictions], dim=-1)
ground_t = t.cat([test_tokens['input_ids'], t.tensor([[0]], device=device)], dim=-1)
rprint(f'correct/tot = {t.sum(predictions == ground_t).int()}/{len(test_tokens['input_ids'].squeeze())}')


loss = 4.347401142120361

correct/tot = 35/111

#### Activations

In [13]:
test_str = "Hello world, I'm John"

wrapped_m = StandardizedTransformer(MODEL_NAME, enable_attention_probs=True)

In [ ]:
with wrapped_m.trace(test_str):
    pattern_0 = wrapped_m.attention_probabilities[0].save()

attention_pattern = pattern_0[0]
str_tokens = wrapped_m.tokenizer.tokenize(test_str)

print(f"Pattern Shape: {attention_pattern.shape}") # Should be [12, seq_len, seq_len]

# 4. Display using CircuitsVis
display(cv.attention.attention_patterns(
    tokens=str_tokens,
    attention=attention_pattern,
))

Pattern Shape: torch.Size([12, 6, 6])


#### Finding induction heads

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [25]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")

cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,
    positional_embedding_type="shortformer",
)

hooked_model = HookedTransformer(cfg)

from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)


In [21]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [13]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

In [ ]:
logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [37]:
LAYERS = 2

for i in range(0, LAYERS):
  display(cv.attention.attention_patterns(
      tokens=model.to_str_tokens(text),
      attention=cache['pattern', i],
  ))